In [1]:
from gammapy.datasets import MapDataset, MapDatasetEventSampler
from gammapy.modeling.models import SkyModel
from astropy.coordinates import SkyCoord, SkyOffsetFrame
from astropy import units as u
from gammapy.data import DataStore
import numpy as np

In [2]:
data_store = DataStore.from_dir("$GAMMAPY_DATA/hess-dl3-dr1")

In [3]:
obs = data_store.obs(23523)

In [4]:
events = obs.events

In [5]:
frame = SkyOffsetFrame(origin=obs.pointing_radec)
pseudo_fov_coord = events.radec.transform_to(frame)
fov_lon = pseudo_fov_coord.lon
fov_lat = pseudo_fov_coord.lat

In [6]:
bkg_values = obs.bkg.evaluate(
    energy=events.energy,
    fov_lon=fov_lon,
    fov_lat=fov_lat,
)

/Users/adonath/software/mambaforge/envs/jolideco-dev/lib/python3.9/site-packages/gammapy/utils/interpolation.py:185: RuntimeWarning: overflow encountered in exp
  output = np.exp(values)


In [7]:
aeff_values = obs.aeff.evaluate(
    energy_true=events.energy,
    offset=events.offset
)
exp_values = aeff_values * obs.observation_live_time_duration

In [8]:
model = SkyModel.create("pl", "gauss")
model.spatial_model.frame = "galactic"

In [9]:
flux = model.evaluate(
    lon=events.galactic.l,
    lat=events.galactic.b,
    energy=events.energy,
)

In [10]:
rad = np.linspace(0, )

obs.psf.evaluate(
    rad=
    offset=events.offset,
    energy_true=events.energy,
)

SyntaxError: invalid syntax (172012423.py, line 5)

In [54]:
# What we need is a sparse PSFKernelMap, i.e. one 3d kernel per position
# What we need is a sparse EDispMap, i.e. one 1d kernel per position
# What we need is a sparse Aeff, i.e. one 3d cube per position
# However PSF, EDisp and aeff could be combined 
# that's the only way to get it fast...

In [11]:
(40 * 40 * 10_000 * 32 * u.bit).to("Mbyte")

<Quantity 64. Mbyte>

In [ ]:
class EventResponse:
    """Reponse per event"""
    def __init__(self, weights, coords):
        self.weights = weights
        self.coords = coords
        
        
        

In [47]:
# weights per event can be pre computed, this should happen in the EventDataseMaker
#weights = psf_kernel_3d * edisp_1d * exp_3d


# Think of it as:
# - each event has it's own 3d cube and associated set of coordinates
# - it would be beautiful to use a Map for this, however then we need
# a loop over events (and their associated maps....this will never be efficient...)
# we do not have the resources to store all IRF independently either so we 
# have to combine them into precomputed weights...

# if we use a cube for each event of 10 x 40 x 40 and 32 bit we use ~680 mb for 10_000
# events this should be acceptable, however we cannot do oversampling...
print(((10 * 40 * 40 * 10_000) * 32 * u.bit).to("Mbyte"))

# we could use radial symmetric integration for each event:
print(((10 * 100 * 10_000) * 32 * u.bit).to("Mbyte"))

# hoever we still need the coordinate grids:
# 100 * 10_000 for the spatial coordinates
# 10 * 10_000 for the spectral coordinates

# the data structure for the weights currently does not exist
# maybe we introduce a EventResponse? 


# weights are either (event_idx, energy_true, rad)
# or (event_idx, energy_true, lon, lat)

# each event has different lon / lat and energy range
npred = (model(x, y, E) * weights).sum()

640.0 Mbyte
40.0 Mbyte


NameError: name 'x' is not defined

In [48]:
(100 * 10_000 * 32 * u.bit).to("Mbyte")

<Quantity 4. Mbyte>

In [49]:
dataset = MapDataset.read(
    "$GAMMAPY_DATA/cta-1dc-gc/cta-1dc-gc.fits.gz", name="cta_dataset"
)


In [51]:
kernel = dataset.edisp.get_edisp_kernel()

In [55]:
model = SkyModel.create("pl", "gauss")

In [133]:
n_events = 1000
n_lon = 40
n_energy = 10

energy_sparse = np.geomspace(1, 10, n_energy) * u.TeV
energy = (energy_sparse * np.ones(n_events)[:, np.newaxis]).reshape((-1, 10, 1, 1))

lon_sparse = np.linspace(-1, 1, n_lon) * u.deg
lat_sparse = np.linspace(-1, 1, n_lon) * u.deg

lon, lat = np.meshgrid(lon_sparse, lat_sparse)

lon = (lon * np.ones(n_events)[:, np.newaxis, np.newaxis]).reshape((-1, 1, 40, 40))
lat = (lat * np.ones(n_events)[:, np.newaxis, np.newaxis]).reshape((-1, 1, 40, 40))

In [126]:
weights = np.ones((n_events, len(energy_sparse), len(lon_sparse), len(lat_sparse)), dtype=np.float32)

In [127]:
%%timeit

flux = model(lon=lon, lat=lat, energy=energy)
npred = (flux * weights).sum(axis=(1, 2, 3)) # + bkg 

93.5 ms ± 684 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [128]:
(weights.nbytes * u.byte).to("Mbyte")

<Quantity 64. Mbyte>

In [129]:
(lon.nbytes * u.byte).to("Mbyte")

<Quantity 12.8 Mbyte>

In [130]:
(lat.nbytes * u.byte).to("Mbyte")

<Quantity 12.8 Mbyte>

In [131]:
(energy.nbytes * u.byte).to("Mbyte")

<Quantity 0.08 Mbyte>

In [ ]:
# 1 sec per 10_000 events

In [121]:
%%time
dataset.npred()

Position <SkyCoord (ICRS): (ra, dec) in deg
    (0., 0.)> is outside valid IRF map range, using nearest IRF defined within
Position <SkyCoord (ICRS): (ra, dec) in deg
    (0., 0.)> is outside valid IRF map range, using nearest IRF defined within


CPU times: user 135 ms, sys: 18.7 ms, total: 154 ms
Wall time: 151 ms


WcsNDMap

	geom  : WcsGeom 
 	axes  : ['lon', 'lat', 'energy']
	shape : (320, 240, 10)
	ndim  : 3
	unit  : 
	dtype : float64

In [136]:
k = dataset.psf.get_psf_kernel(geom=dataset.exposure.geom)

In [137]:
k.psf_kernel_map

WcsNDMap

	geom  : WcsGeom 
 	axes  : ['lon', 'lat', 'energy_true']
	shape : (29, 29, 14)
	ndim  : 3
	unit  : 
	dtype : float64

In [13]:
100 * 100 * 16

160000

In [35]:
from gammapy.maps import WcsGeom, MapAxis

In [39]:
n_events = 100
event_axis = MapAxis.from_nodes(np.arange(n_events), name="event-idx")

In [40]:
event_axis

MapAxis

	name       : event-idx 
	unit       : ''        
	nbins      : 100       
	node type  : center    
	center min : 0.0e+00   
	center max : 9.9e+01   
	interp     : lin       

In [47]:
from gammapy.datasets.evaluator import MapEvaluator

In [48]:
evaluators = {}

for model in models:
    evaluators[model.name] = MapEvaluator(model, exposure)

In [ ]:
class FluxEvaluator:
    """"""
    def __init__(self, geom_true):
        pass
    
    def compute_flux(self):
        pass
    

In [46]:
from gammapy.irf.core import IRF

class EventIRFKernelMap(IRF):
    tag = "evemnt-irf-kernel-map"
    required_axes = ["event_id", "energy_true", "wcs"]
    

In [42]:
flux_cube = 

ValueError: object too deep for desired array